In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random, json

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

In [ ]:
from src.pipeline.captioning_scratch import ImageCaptionerScratch
from src.pipeline.captioning_keras import ImageCaptionerKeras
from src.evaluation.metrics import compute_bleu4, timed_predict_from_features

VOCAB_PATH         = '../../data/captions/vocab.json'
TEST_CAPTIONS_PATH = '../../data/captions/test_captions.json'
TEST_FEATURES_PATH = '../../data/features/test_features.npy'

with open(TEST_CAPTIONS_PATH) as f:
    test_data = json.load(f)
test_image_names = list(test_data.keys())
test_references  = [test_data[name] for name in test_image_names]

test_features = np.load(TEST_FEATURES_PATH, allow_pickle=True).item()
if isinstance(test_features, dict):
    test_feat_matrix = np.stack([test_features[n] for n in test_image_names])
else:
    test_feat_matrix = test_features

In [ ]:
BEST_WEIGHTS = '../../weights/rnn_lstm/lstm_preinject_L?_H???.h5'
BEST_TYPE    = 'lstm'

best_captioner = ImageCaptionerKeras(BEST_TYPE)
best_captioner.load_model('InceptionV3', BEST_WEIGHTS, VOCAB_PATH)

In [ ]:
MAX_LEN_VARIANTS = [10, 15, 20, 25, 30]

ablation_results = {}

for max_len in MAX_LEN_VARIANTS:
    captions, total, avg = timed_predict_from_features(
        best_captioner, test_feat_matrix, max_len=max_len
    )
    bleu4 = compute_bleu4(test_references, captions)
    ablation_results[max_len] = {'bleu4': bleu4, 'avg_time': avg, 'captions': captions}
    print(f'max_len={max_len:2d}:  BLEU-4={bleu4:.4f}  avg_time={avg:.3f}s')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns, os

sns.set_theme(style='whitegrid')
os.makedirs('../../results/plots', exist_ok=True)

lens   = sorted(ablation_results.keys())
bleu4s = [ablation_results[l]['bleu4']    for l in lens]
times  = [ablation_results[l]['avg_time'] for l in lens]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(lens, bleu4s, marker='o', linewidth=2, color='#4C72B0')
axes[0].set_title('BLEU-4 vs max_len', fontsize=14)
axes[0].set_xlabel('max_len', fontsize=12)
axes[0].set_ylabel('BLEU-4', fontsize=12)
axes[0].set_xticks(lens)
for x, y in zip(lens, bleu4s):
    axes[0].annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

axes[1].plot(lens, times, marker='s', linewidth=2, color='#DD8452')
axes[1].set_title('Avg Inference Time vs max_len', fontsize=14)
axes[1].set_xlabel('max_len', fontsize=12)
axes[1].set_ylabel('Time (s)', fontsize=12)
axes[1].set_xticks(lens)

plt.tight_layout()
plt.savefig('../../results/plots/bleu4_vs_maxlen.png', dpi=150, bbox_inches='tight')
plt.show()

import json
with open('../../results/tables/caption_length_ablation.json', 'w') as f:
    json.dump({str(k): {'bleu4': v['bleu4'], 'avg_time': v['avg_time']} for k, v in ablation_results.items()}, f, indent=2)

## Analisis

**Apakah caption lebih panjang selalu menghasilkan BLEU-4 lebih tinggi?**

Tidak selalu. BLEU-4 mengukur precision n-gram — caption yang lebih panjang dari referensi akan mengandung lebih banyak token yang tidak ada di referensi, menurunkan precision. Selain itu, model yang ter-training dengan `max_len=30` cenderung menghasilkan caption yang melenceng setelah token ke-20 karena distribusi probabilitas di ujung sequence menjadi tidak terdistribusi baik.

Titik optimal `max_len` biasanya ada di sekitar panjang rata-rata caption referensi dalam training set.